In [1]:
%pip install pandas numpy
import pandas as pd
import hashlib

passengers = pd.read_pickle("../data/processed/passengers_clean.pkl")
bookings   = pd.read_pickle("../data/processed/bookings_clean.pkl")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
def hash_id(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode()).hexdigest()[:16] 

def mask_email(email):
    if pd.isna(email):
        return None
    local, _, domain = email.partition("@")
    if len(local) <= 2:
        masked_local = local[0] + "*"
    else:
        masked_local = local[0] + "*" * (len(local) - 2) + local[-1]
    return f"{masked_local}@{domain}"

def mask_phone(phone):
    if pd.isna(phone):
        return None
    phone = str(phone)
    return phone[:3] + "*" * (len(phone) - 5) + phone[-2:]

def pseudonymize_name(passenger_id, salt="asg2026"):
    
    idx = int(hashlib.sha256((str(passenger_id) + salt).encode()).hexdigest(), 16) % 1000
    return f"Passenger_{idx:04d}"

In [ ]:
passengers_masked = passengers.copy()

passengers_masked["passenger_alias"] = passengers_masked["passenger_id"].apply(pseudonymize_name)
passengers_masked["email_masked"] = passengers_masked["email"].apply(mask_email)
passengers_masked["phone_masked"] = passengers_masked["phone"].apply(mask_phone)
passengers_masked["aadhaar_hash"] = passengers_masked["aadhaar_id"].apply(hash_id)
passengers_masked["birth_year"] = pd.to_datetime(passengers_masked["date_of_birth"]).dt.year
passengers_masked["age_band"] = pd.cut(
    passengers_masked["age"],
    bins=[0, 18, 30, 45, 60, 120],
    labels=["<18", "18-29", "30-44", "45-59", "60+"]
)

passengers_analytics = passengers_masked[[
    "passenger_id", "passenger_alias", "gender", "age_band",
    "email_masked", "phone_masked", "aadhaar_hash", "birth_year"
]]

print(passengers_analytics.head())

  passenger_id passenger_alias gender age_band                 email_masked  \
0        P1000  Passenger_0112      F    45-59  v***************e@gmail.com   
1        P1001  Passenger_0751      M      <18    k***********y@hotmail.com   
2        P1002  Passenger_0418      M      60+       m********u@outlook.com   
3        P1003  Passenger_0804      F      60+      m*********a@hotmail.com   
4        P1004  Passenger_0556      M    18-29  s*************e@outlook.com   

     phone_masked      aadhaar_hash  birth_year  
0  +91*********90  99466baa9b8fe69a        1974  
1  +91*********97  6d05ccbeb5011dd5        2011  
2  +91*********92  3e24ac79e2c95679        1954  
3  +91*********51  4e880723014ee24b        1965  
4  +91*********13  3ca09c31f18da71a        2005  


In [4]:
bookings_masked = bookings.copy()
bookings_masked["passport_hash"] = bookings_masked["passport_number"].apply(hash_id)

bookings_analytics = bookings_masked.drop(columns=[
    "passport_number", "emergency_contact_name", "emergency_contact_phone"
])

print(bookings_analytics.head())

  booking_id passenger_id flight_id            booking_date     status  \
0      B1000        P1591     AI192 2025-06-14 11:37:36.951  CANCELLED   
1      B1001        P1803     6F026 2025-11-02 11:37:36.951  CANCELLED   
2      B1002        P1083     SJ010 2025-08-25 11:37:36.951  CANCELLED   
3      B1003        P1364     AI069 2025-12-30 11:37:36.951  CONFIRMED   
4      B1004        P1885     UK003 2025-10-02 11:37:36.951    PENDING   

  seat_number     passport_hash  
0          3D  59e7e7738c8711b2  
1         18A  6a95f336b866dc4d  
2         30C  8d413701d70b5174  
3         33A  19c4a665e8bc1266  
4         25C  d9157c803314e083  


In [ ]:
passengers_analytics.to_pickle("../data/processed/passengers_analytics.pkl")
bookings_analytics.to_pickle("../data/processed/bookings_analytics.pkl")


passengers_analytics.to_csv("../data/processed/passengers_analytics.csv", index=False)
bookings_analytics.to_csv("../data/processed/bookings_analytics.csv", index=False)